# KosenMap Website 取扱説明書 —— はじめに

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** どこから読み、最初に何をするか |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

## 1. 準備(この PC で一度だけ)

| 要るもの | 確かめ方 / 入れ方 |
|---|---|
| VS Code と **Jupyter 拡張**(`ms-toolsai.jupyter`) | 拡張機能の一覧に「Jupyter」がある |
| **Python 3.13 と ipykernel** | `python -m pip install --user ipykernel` |
| **PowerShell 7**(`pwsh`) | `pwsh --version` |
| SSH の鍵 `~\.ssh\km_ops`(配備・`%%host`・控え)と `~\.ssh\km_vps`(`sudo` のセル) | 下の「環境の確認」で見る |
| `php`(`check.php` を回す) | `php -v` |
| OpenSSL(添付の復号) | `openssl version` |

## 2. 開き方

1. VS Code で `server/docs/00-start.ipynb`(このファイル)を開く
2. 右上の **「カーネルの選択」→ Python 3.13** を選ぶ
3. **下の最初のコードセルを実行する**(`Shift+Enter`)。
   `使えるセル: %%ps(手元) / %%host(ホスト) / %%terminal(別の窓)` と出れば準備完了

> ノートブックごとに、先頭のこのセルを1回実行する(カーネルはノートブックごとに別)。

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()

## 3. 印の意味

| 印 | 意味 |
|---|---|
| 🟢 | **読むだけ。** 何も変えない。迷ったらここから |
| 🟡 | **手元が変わる。** この PC にファイルを作る・登録する。本番には触れない |
| 🔴 | **本番が変わる。** 実行前に `yes` の入力を求める |
| 🔑 | **別の窓で開く。** sudo のパスワードなど対話が要るもの |

**🔴 のセルは、実行すると入力欄が出る。** `yes` と打つまで何もしない。
それ以外を打つか空のまま Enter すれば「中断しました(何も実行していません)」で終わる。

## 4. セルの型

| 1行目 | どこで動くか | 作業場所 |
|---|---|---|
| `%%ps` | この PC の **PowerShell 7** | `server/scripts` |
| `%%host` | **ホストの sh**(`kmops` の権限。docker は使える。**sudo は使えない** —— sudo が要るものは `%%terminal` で `km` から) | `/opt/kosenmap` |
| `%%terminal` | **新しい PowerShell の窓**(sudo のパスワード入力など) | `server/scripts` |

1行目に付けられるもの:

| 選択肢 | 意味 |
|---|---|
| `--confirm "文"` | 実行前に `yes` を求める。**本番を変えるセルには必ず付いている**(`check.php` が見張る) |
| `--timeout 秒` | 過ぎたら止める。`%%host` は既定 600 秒、`%%ps` は既定で無制限 |

**止めたいとき**は、セルの左の ■(中断)を押す。実行中のプロセスも止まる
(ホスト側で動き始めた処理は残ることがある)。

## 5. 気をつけること

- **出力はノートブックに保存される。** 人に渡す前に「すべての出力をクリア」する
  (秘密を表示するセルは置いていないが、ホストの状態は出る)
- 接続先を変えたいときは、読み込む前に環境変数 `KM_HOST` / `KM_USER` / `KM_KEY` / `KM_REMOTE_PATH` を置く
- **`.ipynb` を直したら `check.php` を回す**(`notebooks` の節が、印と確認の付け忘れを見る)

### 環境の確認(この PC)

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
"PowerShell  : $($PSVersionTable.PSVersion)"
foreach ($c in 'ssh', 'scp', 'php', 'openssl', 'tar') {
    $g = Get-Command $c -ErrorAction SilentlyContinue | Select-Object -First 1
    "{0,-11} : {1}" -f $c, $(if ($g) { $g.Source } else { '★ 見つかりません' })
}
foreach ($k in 'km_ops', 'km_vps', 'km_backup') {
    $key = "$env:USERPROFILE\.ssh\$k"
    $use = switch ($k) { 'km_ops' { '配備・%%host・控え(kmops)' } 'km_vps' { 'sudo のセル(km)' } default { '週次の控え(門番付き)' } }
    "SSH の鍵    : {0,-10} {1} {2}" -f $k, $(if (Test-Path $key) { 'あり' } else { '★ ありません' }), $use
}
# パスフレーズ付きの鍵は ssh-agent に載っていないと BatchMode のセルが黙って失敗する(12 §7-3)
$agent = Get-Service ssh-agent -ErrorAction SilentlyContinue
"ssh-agent   : $(if ($agent) { "$($agent.Status) / 載っている鍵 $((@(ssh-add -l 2>$null)).Count) 本" } else { '★ ありません' })"
. .\backup-lib.ps1
try { "控えの置き場: $(Get-KmBackupRoot)" } catch { "控えの置き場: ★ $($_.Exception.Message)" }
$task = Get-ScheduledTask -TaskName 'KosenMap バックアップ(週次)' -ErrorAction SilentlyContinue
"週次タスク  : $(if ($task) { "$($task.State) / 次回 $(($task | Get-ScheduledTaskInfo).NextRunTime)" } else { '★ 登録されていません' })"

### ホストへ繋がるか

`.env` を docker compose が読めるかも見る。**読めないとメールもバックアップも全部落ちる**
(2026-09-13 に、合言葉へ引用符を入れて実際に起きた)。エラー文には値の断片が出るので、ここでは出さない。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
echo "ホスト : $(hostname) / $(date '+%Y-%m-%d %H:%M:%S %Z')"
echo "置き場 : $(pwd)"
if docker compose config -q >/dev/null 2>&1; then
  echo ".env   : docker compose が読めます"
else
  echo ".env   : ★ docker compose が読めません(値に引用符・\\・\$ が入っていないか)"
fi
echo
docker ps --format '{{.Names}}\t{{.Status}}' | sort